# 🔍 Model Investigation: Why Low Risk for Low Battery Device?

**Date:** 2025-10-14  
**Device:** 861275072344001  
**Issue:** Battery at 2.78V showing only 3% failure risk (expected >80%)

## Hypotheses to Test:
1. Model uses rolling mean (7-day average), not current battery value
2. Train/test split is random, not temporal (data leakage possible)
3. Recent failures (this week) may not be in training data
4. Missing 'battery_voltage' raw feature (only have rolling features)

In [ ]:
# Setup
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from src.core.logic import load_data

# Load model
model_payload = joblib.load('model.joblib')
model = model_payload['model']
features = model_payload['features']

print(f"Model loaded: {len(features)} features")
print(f"Features: {features}")

## 1. Load and Inspect Data

In [ ]:
# Load recent data (last 200k rows for speed)
print("Loading data (this may take a moment)...")
df_sample = pd.read_csv('data/processed/payloads_processed.csv',
                        parse_dates=['@timestamp'],
                        nrows=200000)

print(f"✓ Loaded {len(df_sample)} rows")
print(f"  Date range: {df_sample['@timestamp'].min()} to {df_sample['@timestamp'].max()}")
print(f"  Devices: {df_sample['device_id'].nunique()}")

# Filter to recent data (90 days)
max_date = df_sample['@timestamp'].max()
cutoff = max_date - pd.Timedelta(days=90)
df_recent = df_sample[df_sample['@timestamp'] >= cutoff].copy()

print(f"\n✓ Recent data (90 days): {len(df_recent)} rows, {df_recent['device_id'].nunique()} devices")

## 2. Inspect Problem Device: 861275072344001

In [ ]:
# Get data for problem device
device_id = '861275072344001'
device_data = df_recent[df_recent['device_id'] == device_id].copy()
device_data = device_data.sort_values('@timestamp')

print(f"Device {device_id}:")
print(f"  Records: {len(device_data)}")
print(f"  Date range: {device_data['@timestamp'].min()} to {device_data['@timestamp'].max()}")
print(f"\nBattery Statistics:")
print(device_data['battery_voltage'].describe())

# Plot battery over time
plt.figure(figsize=(12, 4))
plt.plot(device_data['@timestamp'], device_data['battery_voltage'], marker='o', alpha=0.6)
plt.axhline(y=2.5, color='r', linestyle='--', label='Failure Threshold (2.5V)')
plt.axhline(y=2.78, color='orange', linestyle='--', label='Current Value (2.78V)')
plt.xlabel('Date')
plt.ylabel('Battery Voltage (V)')
plt.title(f'Battery Voltage Over Time - Device {device_id}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Check last 30 days
last_30 = device_data[device_data['@timestamp'] >= device_data['@timestamp'].max() - pd.Timedelta(days=30)]
print(f"\nLast 30 days:")
print(f"  Mean battery: {last_30['battery_voltage'].mean():.2f}V")
print(f"  Min battery: {last_30['battery_voltage'].min():.2f}V")
print(f"  Std battery: {last_30['battery_voltage'].std():.3f}V")

## 3. Feature Engineering - Compute Rolling Features

In [ ]:
# Simplified feature engineering for this device
device_data = device_data.set_index('@timestamp').sort_index()

# Battery rolling features
device_data['battery_rolling_mean_7d'] = device_data['battery_voltage'].rolling('7D').mean()
device_data['battery_rolling_std_7d'] = device_data['battery_voltage'].rolling('7D').std()

# Plot rolling vs actual
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Actual vs rolling mean
ax1.plot(device_data.index, device_data['battery_voltage'], label='Actual Battery', alpha=0.7)
ax1.plot(device_data.index, device_data['battery_rolling_mean_7d'], label='7-day Rolling Mean', linewidth=2)
ax1.axhline(y=2.5, color='r', linestyle='--', label='Failure Threshold')
ax1.set_ylabel('Voltage (V)')
ax1.set_title('Actual vs Rolling Mean Battery Voltage')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Rolling std
ax2.plot(device_data.index, device_data['battery_rolling_std_7d'], color='orange', label='7-day Rolling Std')
ax2.set_ylabel('Std Dev (V)')
ax2.set_xlabel('Date')
ax2.set_title('Battery Voltage Variability')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show latest values
latest = device_data.iloc[-1]
print(f"\n📊 Latest Feature Values (what model sees):")
print(f"  battery_voltage (actual): {latest['battery_voltage']:.2f}V")
print(f"  battery_rolling_mean_7d: {latest['battery_rolling_mean_7d']:.2f}V")
print(f"  battery_rolling_std_7d: {latest['battery_rolling_std_7d']:.3f}V")
print(f"\n⚠️ INSIGHT: Model sees rolling mean ({latest['battery_rolling_mean_7d']:.2f}V), NOT current value ({latest['battery_voltage']:.2f}V)!")

## 4. Check Train/Test Split Strategy

In [ ]:
# Simulate train_test_split to understand data distribution
from sklearn.model_selection import train_test_split

# Check if we have future_failure column in sample data
if 'future_failure' in df_recent.columns:
    # Random split (current approach)
    train_random, test_random = train_test_split(
        df_recent, 
        test_size=0.2, 
        random_state=42,
        stratify=df_recent['future_failure'] if df_recent['future_failure'].notna().all() else None
    )
    
    print("📊 RANDOM SPLIT (current approach):")
    print(f"  Train: {len(train_random)} rows ({train_random['@timestamp'].min()} to {train_random['@timestamp'].max()})")
    print(f"  Test:  {len(test_random)} rows ({test_random['@timestamp'].min()} to {test_random['@timestamp'].max()})")
    print(f"  ⚠️ Test data includes dates from entire range (data leakage risk!)")
    
    # Temporal split (recommended)
    df_sorted = df_recent.sort_values('@timestamp')
    split_idx = int(len(df_sorted) * 0.8)
    train_temporal = df_sorted.iloc[:split_idx]
    test_temporal = df_sorted.iloc[split_idx:]
    
    print(f"\n📊 TEMPORAL SPLIT (recommended):")
    print(f"  Train: {len(train_temporal)} rows ({train_temporal['@timestamp'].min()} to {train_temporal['@timestamp'].max()})")
    print(f"  Test:  {len(test_temporal)} rows ({test_temporal['@timestamp'].min()} to {test_temporal['@timestamp'].max()})")
    print(f"  ✓ Test data is strictly future data (no leakage)")
    
    # Check if problem device is in test set
    in_test_random = device_id in test_random['device_id'].values
    in_test_temporal = device_id in test_temporal['device_id'].values
    
    print(f"\nProblem device {device_id}:")
    print(f"  In random test set: {in_test_random}")
    print(f"  In temporal test set: {in_test_temporal}")
else:
    print("⚠️ 'future_failure' column not in sample data - cannot simulate split")
    print("This column is created during training by looking ahead 30 days")

## 5. Target Distribution Analysis

In [ ]:
if 'future_failure' in df_recent.columns:
    failure_rate = df_recent['future_failure'].mean() * 100
    print(f"📊 Target Distribution:")
    print(f"  Failure rate: {failure_rate:.2f}%")
    print(f"  No-failure: {(100-failure_rate):.2f}%")
    print(f"  Total samples: {len(df_recent)}")
    
    # Plot distribution
    plt.figure(figsize=(8, 5))
    df_recent['future_failure'].value_counts().plot(kind='bar')
    plt.xlabel('Future Failure (0=No, 1=Yes)')
    plt.ylabel('Count')
    plt.title('Target Distribution: Future Failure (30 days ahead)')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Check battery distribution for failures vs non-failures
    fig, ax = plt.subplots(figsize=(10, 5))
    df_recent[df_recent['future_failure'] == 0]['battery_voltage'].hist(bins=50, alpha=0.5, label='No Failure', ax=ax)
    df_recent[df_recent['future_failure'] == 1]['battery_voltage'].hist(bins=50, alpha=0.5, label='Failure', ax=ax)
    ax.axvline(x=2.5, color='r', linestyle='--', label='Threshold (2.5V)')
    ax.set_xlabel('Battery Voltage (V)')
    ax.set_ylabel('Frequency')
    ax.set_title('Battery Distribution: Failures vs Non-Failures')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Cannot analyze target - 'future_failure' not in data")
    print("\nTo create this column, we need to:")
    print("1. Group by device_id")
    print("2. For each row, check if battery < 2.5V in next 30 days")
    print("3. Label current row as future_failure=1 if true")

## 6. Recommendations

### Critical Issues Found:

1. **Missing Raw Battery Feature**  
   - Model only sees `battery_rolling_mean_7d` (~3.1V), not actual current value (2.78V)
   - **Fix:** Add `battery_voltage` as a raw feature

2. **Random Train/Test Split**  
   - Current split mixes past and future data randomly
   - Cannot properly validate on "unseen future" data
   - **Fix:** Implement temporal split (train on older 80%, test on recent 20%)

3. **Recent Failures Not in Training**  
   - If Enzo's failures happened this week (Oct 2025), they may be in test set or not in 90-day window
   - **Fix:** Retrain with full dataset (`--all`) and temporal split

### Next Steps:

1. Add `battery_voltage` raw feature to training script
2. Implement temporal train/test split option
3. Retrain with recent failure data
4. Re-evaluate on problem devices

## 7. Python vs Jupyter Trade-offs

### ✅ Python Scripts (.py) - Best for:
- **Production training pipelines** (reproducible, versionable)
- **CI/CD integration** (automated testing, deployment)
- **Long-running jobs** (no kernel crashes)
- **Code review** (clear diffs in git)

### ✅ Jupyter Notebooks (.ipynb) - Best for:
- **Exploratory Data Analysis** (like this investigation!)
- **Debugging models** (inspect intermediate values)
- **Visualizations** (interactive plots)
- **Sharing insights** (markdown + code + outputs)
- **Hypothesis testing** (quick iterations)

### 🎯 Recommended Workflow:
1. **Explore in Jupyter** → Understand data, test features, debug issues
2. **Productionize in Python** → Clean scripts for training, monitoring
3. **Document in Jupyter** → Analysis reports, model evaluation

This investigation proves Jupyter's value - we quickly identified the rolling mean issue by visualizing the data!